In [87]:
import sqlite3
import pandas as pd
import numpy as np 

con = sqlite3.connect('database.db')
cur = con.cursor()

In [88]:
instruments_df = pd.read_sql_query(f"SELECT * FROM instrument", con)


In [89]:
def get_d3_domains(instrument_id):
    df = pd.read_sql_query(f"""SELECT * FROM pricedata WHERE instrument_id = {instrument_id} ORDER BY time""", con, parse_dates=['time'])

    df['time_shifted'] = df.time.shift(-1)
    df['time_difference'] = df.time_shifted - df.time
    df['has_time_gap'] = 0
    most_frequent_time_difference = df['time_difference'].value_counts().iloc[[0]].index
    df.loc[df.time_difference != most_frequent_time_difference[0], 'has_time_gap'] = 1

    df['domain_id'] = 0
    gap_indexes = df[df.has_time_gap == 1].index
    for i in range(gap_indexes.shape[0]): df.loc[gap_indexes[i], 'domain_id'] = i + 1 

    df.loc[:, 'domain_id'] = df.domain_id.replace(0, np.nan).bfill().astype(int)

    domain_df = df.groupby('domain_id').agg({'time': ['min', 'max']})
    domain_df.columns = ['time_min', 'time_max']
    return df, domain_df

In [90]:
for instrument_id in instruments_df.id:
    df, domain_df = get_d3_domains(instrument_id)
    for domain_id, group in df.groupby('domain_id'):
        res = cur.execute("INSERT INTO domain (instrument_id, time_min, time_max) VALUES (?, ?, ?)", (instrument_id, domain_df.iloc[0].time_min.timestamp(), domain_df.iloc[0].time_max.timestamp()))
        stored_domain_id = res.lastrowid
        for item in group.itertuples():
            cur.execute(f"UPDATE pricedata SET domain_id = {stored_domain_id} WHERE id = {item.id}")
        con.commit()